Agentic AI System

In [3]:
!pip install -q langgraph pydantic

In [4]:
from typing import Dict, List
from pydantic import BaseModel
from langgraph.graph import StateGraph

Shared Deal State

In [5]:
class DealState(BaseModel):
    # ============ INPUTS ============
    # Target company
    target_revenue: float
    target_net_income: float

    # Acquirer company
    acquirer_revenue: float
    acquirer_net_income: float
    acquirer_shares_outstanding: float   # in millions
    acquirer_share_price: float          # current market price

    # Deal terms
    deal_price: float                    # total purchase consideration
    cash_pct: float = 0.5                # proposed financing mix
    stock_pct: float = 0.5

    # Financial assumptions
    cost_of_debt_pretax: float = 0.05    # 5% pretax interest rate on new debt
    tax_rate: float = 0.25               # 25% effective tax rate
    synergies_annual: float = 0.0        # expected annual cost/revenue synergies

    # Qualitative inputs
    management_list: List[str] = []
    shareholder_data: Dict[str, str] = {}
    employee_incentive_programs: Dict[str, float] = {}
    industries_involved: List[str] = []
    countries_involved: List[str] = []
    party_priorities: Dict[str, List[str]] = {}
    financial_flexibility: float = 0.0

    # ============ AGENT OUTPUTS ============
    # Financial model
    combined_revenue: float = 0.0
    new_debt: float = 0.0
    new_shares_issued: float = 0.0
    interest_expense_aftertax: float = 0.0
    synergies_aftertax: float = 0.0
    pro_forma_net_income: float = 0.0
    pro_forma_shares: float = 0.0
    pro_forma_eps: float = 0.0
    standalone_eps: float = 0.0

    # Accretion / Dilution
    accretion_dilution_pct: float = 0.0
    accretion_dilution_result: str = ""
    accretion_dilution_magnitude: str = ""

    # Capital structure recommendation
    recommended_cash_pct: float = 0.0
    recommended_stock_pct: float = 0.0
    recommended_pro_forma_eps: float = 0.0
    recommended_accretion_pct: float = 0.0
    deal_timeline_months: int = 0

    # Stakeholders
    key_management_to_retain: List[str] = []
    incentive_risk: str = ""
    potential_conflicts: List[str] = []

    # Regulatory
    antitrust_risk: str = ""
    approvals_required: List[str] = []
    cross_border_flag: bool = False

    # Negotiation
    negotiation_strategy: str = ""
    fallback_option: str = ""

Finanical Model Agent

In [6]:
def analyze_financial_statements(state: DealState) -> DealState:
    # Combined revenue (additive; revenue synergies/dis-synergies abstracted out for this demo)
    state.combined_revenue = state.target_revenue + state.acquirer_revenue

    # Financing breakdown
    state.new_debt = state.deal_price * state.cash_pct
    stock_consideration = state.deal_price * state.stock_pct
    state.new_shares_issued = stock_consideration / state.acquirer_share_price

    # After-tax interest expense on new debt raised to fund the cash portion
    interest_pretax = state.new_debt * state.cost_of_debt_pretax
    state.interest_expense_aftertax = interest_pretax * (1 - state.tax_rate)

    # After-tax synergies
    state.synergies_aftertax = state.synergies_annual * (1 - state.tax_rate)

    # Pro forma net income
    state.pro_forma_net_income = (
        state.target_net_income
        + state.acquirer_net_income
        + state.synergies_aftertax
        - state.interest_expense_aftertax
    )

    # Pro forma share count
    state.pro_forma_shares = state.acquirer_shares_outstanding + state.new_shares_issued

    # EPS — proper formula: Net Income / Shares Outstanding
    state.pro_forma_eps = state.pro_forma_net_income / state.pro_forma_shares
    state.standalone_eps = state.acquirer_net_income / state.acquirer_shares_outstanding

    return state

Accretion/Dilution Agent

In [7]:
def model_accretion_dilution(state: DealState) -> DealState:
    pct = (state.pro_forma_eps / state.standalone_eps) - 1
    state.accretion_dilution_pct = pct
    state.accretion_dilution_result = "Accretive" if pct >= 0 else "Dilutive"

    abs_pct = abs(pct)
    if abs_pct >= 0.10:
        state.accretion_dilution_magnitude = "Strongly"
    elif abs_pct >= 0.03:
        state.accretion_dilution_magnitude = "Modestly"
    else:
        state.accretion_dilution_magnitude = "Marginally"

    return state

Capital Structure Optmization Agent

In [8]:
def calculate_optimal_capital_structure(state: DealState) -> DealState:
    best_mix = (state.cash_pct, state.stock_pct)
    best_eps = -float("inf")

    for cash_pct_int in range(0, 101, 10):
        cash_pct = cash_pct_int / 100
        stock_pct = 1 - cash_pct

        # Recompute pro forma EPS under this candidate mix
        new_debt = state.deal_price * cash_pct
        interest_at = new_debt * state.cost_of_debt_pretax * (1 - state.tax_rate)
        new_shares = (state.deal_price * stock_pct) / state.acquirer_share_price

        pf_ni = (
            state.target_net_income
            + state.acquirer_net_income
            + state.synergies_aftertax
            - interest_at
        )
        pf_shares = state.acquirer_shares_outstanding + new_shares
        pf_eps = pf_ni / pf_shares

        if pf_eps > best_eps:
            best_eps = pf_eps
            best_mix = (cash_pct, stock_pct)

    state.recommended_cash_pct = best_mix[0]
    state.recommended_stock_pct = best_mix[1]
    state.recommended_pro_forma_eps = best_eps
    state.recommended_accretion_pct = (best_eps / state.standalone_eps) - 1

    # Timeline: base 4 months, +2 for mega-deals, +2 for cross-border
    months = 4
    if state.deal_price > 5_000_000_000:
        months += 2
    if len(set(state.countries_involved)) > 1:
        months += 2
    state.deal_timeline_months = months

    return state

Stakeholder Analysis Agent

In [9]:
def analyze_stakeholders(state: DealState) -> DealState:
    conflicts = []
    prefs = set(state.shareholder_data.values())

    # Mixed preferences in the cap table
    if "cash" in prefs and "stock" in prefs:
        conflicts.append("Mixed shareholder payout preferences (cash vs. stock)")

    # Recommended structure vs. shareholder preference
    if "cash" in prefs and state.recommended_stock_pct > 0.6:
        conflicts.append("Cash-preferring holders may resist heavy stock consideration")
    if "stock" in prefs and state.recommended_cash_pct > 0.6:
        conflicts.append("Stock-preferring holders may resist heavy cash consideration")

    # Top management to retain (top 3 in the management roster)
    state.key_management_to_retain = state.management_list[:3]

    # Retention risk scaled to number of incentive programs in place
    num_programs = len(state.employee_incentive_programs)
    if num_programs == 0:
        state.incentive_risk = "High — no retention vehicles in place"
    elif num_programs < 2:
        state.incentive_risk = "Medium — limited retention vehicles"
    else:
        state.incentive_risk = "Low — multiple retention vehicles available"

    state.potential_conflicts = conflicts
    return state

Regulatory Compliance Agent

In [10]:
def analyze_regulatory_compliance(state: DealState) -> DealState:
    approvals = []
    countries = set(state.countries_involved)
    us_involved = bool(countries & {"USA", "US", "United States"})
    eu_countries = {"Germany", "France", "Italy", "Spain", "Netherlands", "EU"}
    eu_involved = bool(countries & eu_countries)

    # US merger filings
    if us_involved:
        approvals.append("HSR filing (DOJ Antitrust Division / FTC)")

    # EU Merger Regulation
    if eu_involved:
        approvals.append("EU Commission (Merger Regulation)")

    # CFIUS for foreign-into-US transactions
    foreign_into_us = us_involved and any(
        c not in {"USA", "US", "United States"} for c in countries
    )
    if foreign_into_us:
        approvals.append("CFIUS (national security review)")
        state.cross_border_flag = True

    # Multi-jurisdiction flag
    if len(countries) > 1:
        state.cross_border_flag = True

    # Industry-driven antitrust exposure
    concentrated = {"Technology", "Telecommunications", "Healthcare", "Airlines", "Pharmaceuticals"}
    overlap = [i for i in state.industries_involved if i in concentrated]
    if overlap:
        state.antitrust_risk = f"Elevated — concentrated industry exposure ({', '.join(overlap)})"
    else:
        state.antitrust_risk = "Standard — no immediate concentration concerns"

    if not approvals:
        approvals.append("Local jurisdiction filings (TBD)")

    state.approvals_required = approvals
    return state

Negotiation Strategy Agent

In [11]:
def generate_negotiation_strategy(state: DealState) -> DealState:
    priorities = state.party_priorities.get("acquirer", [])

    if "speed" in priorities:
        state.negotiation_strategy = (
            "Collaborative — prioritize speed-to-close, accept moderate concessions"
        )
    elif "value" in priorities or "price" in priorities:
        state.negotiation_strategy = (
            "Assertive — push on price, structure protections, longer timeline acceptable"
        )
    else:
        state.negotiation_strategy = (
            "Balanced — standard market terms, flexible on non-economic provisions"
        )

    # Fallback depends on financial picture rather than a fixed cutoff
    is_dilutive = state.accretion_dilution_result == "Dilutive"
    low_liquidity = state.financial_flexibility < state.deal_price * 0.5

    if is_dilutive and low_liquidity:
        state.fallback_option = (
            "Shift toward stock consideration to preserve liquidity; "
            "phase synergy realization to offset near-term dilution"
        )
    elif state.financial_flexibility >= state.deal_price:
        state.fallback_option = "Offer cash premium for early close (buy seller certainty)"
    else:
        state.fallback_option = (
            "Add contingent value rights (CVRs) or earnouts to bridge valuation gap"
        )

    return state

Agent Graph

In [12]:
graph = StateGraph(state_schema=DealState)

graph.add_node("Financial_Model_Agent", analyze_financial_statements)
graph.add_node("Accretion_Dilution_Agent", model_accretion_dilution)
graph.add_node("Capital_Structure_Agent", calculate_optimal_capital_structure)
graph.add_node("Stakeholders_Agent", analyze_stakeholders)
graph.add_node("Regulatory_Agent", analyze_regulatory_compliance)
graph.add_node("Negotiation_Agent", generate_negotiation_strategy)

graph.add_edge("Financial_Model_Agent", "Accretion_Dilution_Agent")
graph.add_edge("Accretion_Dilution_Agent", "Capital_Structure_Agent")
graph.add_edge("Capital_Structure_Agent", "Stakeholders_Agent")
graph.add_edge("Stakeholders_Agent", "Regulatory_Agent")
graph.add_edge("Regulatory_Agent", "Negotiation_Agent")

graph.set_entry_point("Financial_Model_Agent")
graph.set_finish_point("Negotiation_Agent")

deal_pipeline = graph.compile()

Demo Run

In [13]:
sample_deal = {
    # Target
    "target_revenue": 1_000_000_000,
    "target_net_income": 80_000_000,

    # Acquirer
    "acquirer_revenue": 5_000_000_000,
    "acquirer_net_income": 500_000_000,
    "acquirer_shares_outstanding": 100_000_000,   # 100M shares
    "acquirer_share_price": 50.0,

    # Deal terms
    "deal_price": 2_000_000_000,
    "cash_pct": 0.5,
    "stock_pct": 0.5,

    # Assumptions
    "cost_of_debt_pretax": 0.05,
    "tax_rate": 0.25,
    "synergies_annual": 50_000_000,

    # Qualitative
    "management_list": ["CEO", "CFO", "CTO", "VP Sales", "VP Engineering"],
    "shareholder_data": {"retail": "cash", "institutional": "stock"},
    "employee_incentive_programs": {"RSUs": 300, "ESPP": 50},
    "industries_involved": ["Technology"],
    "countries_involved": ["USA"],
    "party_priorities": {"acquirer": ["speed", "control"]},
    "financial_flexibility": 1_500_000_000,
}

initial_state = DealState(**sample_deal)
result = deal_pipeline.invoke(initial_state)

Deal Summary

In [14]:
def fmt_money(x: float) -> str:
    if abs(x) >= 1e9:
        return f"${x/1e9:,.2f}B"
    if abs(x) >= 1e6:
        return f"${x/1e6:,.1f}M"
    return f"${x:,.0f}"

def print_summary(r: dict) -> None:
    line = "═" * 64
    print(line)
    print("              M&A DEAL STRUCTURING SUMMARY")
    print(line)
    print()
    print("DEAL OVERVIEW")
    print(f"  Combined Revenue:            {fmt_money(r['combined_revenue'])}")
    print(f"  Deal Price:                  {fmt_money(r['deal_price'])}")
    print(f"  Annual Synergies (pretax):   {fmt_money(r['synergies_annual'])}")
    print()
    print(f"FINANCIAL IMPACT  (Proposed: {r['cash_pct']:.0%} Cash / {r['stock_pct']:.0%} Stock)")
    print(f"  Standalone EPS:              ${r['standalone_eps']:.2f}")
    print(f"  Pro Forma EPS:               ${r['pro_forma_eps']:.2f}")
    print(f"  Accretion / Dilution:        {r['accretion_dilution_pct']:+.1%}  "
          f"({r['accretion_dilution_magnitude']} {r['accretion_dilution_result']})")
    print(f"  New Debt Raised:             {fmt_money(r['new_debt'])}")
    print(f"  New Shares Issued:           {r['new_shares_issued']/1e6:,.1f}M")
    print()
    print("RECOMMENDED STRUCTURE")
    print(f"  Optimal Mix:                 {r['recommended_cash_pct']:.0%} Cash / "
          f"{r['recommended_stock_pct']:.0%} Stock")
    print(f"  Expected Pro Forma EPS:      ${r['recommended_pro_forma_eps']:.2f}")
    print(f"  Expected Accretion:          {r['recommended_accretion_pct']:+.1%}")
    print(f"  Estimated Timeline:          {r['deal_timeline_months']} months")
    print()
    print("STAKEHOLDERS")
    print(f"  Key Management to Retain:    {', '.join(r['key_management_to_retain'])}")
    print(f"  Incentive Risk:              {r['incentive_risk']}")
    conflicts = r['potential_conflicts']
    print(f"  Potential Conflicts:         {conflicts[0] if conflicts else 'None identified'}")
    for c in conflicts[1:]:
        print(f"                               {c}")
    print()
    print("REGULATORY")
    print(f"  Antitrust Risk:              {r['antitrust_risk']}")
    print(f"  Approvals Required:          {r['approvals_required'][0]}")
    for a in r['approvals_required'][1:]:
        print(f"                               {a}")
    print(f"  Cross-Border:                {'Yes' if r['cross_border_flag'] else 'No'}")
    print()
    print("NEGOTIATION")
    print(f"  Strategy:                    {r['negotiation_strategy']}")
    print(f"  Fallback Option:             {r['fallback_option']}")
    print(line)

print_summary(dict(result))

════════════════════════════════════════════════════════════════
              M&A DEAL STRUCTURING SUMMARY
════════════════════════════════════════════════════════════════

DEAL OVERVIEW
  Combined Revenue:            $6.00B
  Deal Price:                  $2.00B
  Annual Synergies (pretax):   $50.0M

FINANCIAL IMPACT  (Proposed: 50% Cash / 50% Stock)
  Standalone EPS:              $5.00
  Pro Forma EPS:               $4.83
  Accretion / Dilution:        -3.3%  (Modestly Dilutive)
  New Debt Raised:             $1.00B
  New Shares Issued:           20.0M

RECOMMENDED STRUCTURE
  Optimal Mix:                 100% Cash / 0% Stock
  Expected Pro Forma EPS:      $5.42
  Expected Accretion:          +8.5%
  Estimated Timeline:          4 months

STAKEHOLDERS
  Key Management to Retain:    CEO, CFO, CTO
  Incentive Risk:              Low — multiple retention vehicles available
  Potential Conflicts:         Mixed shareholder payout preferences (cash vs. stock)
                               

In this project, I built an AI system that simulates how companies evaluate and structure mergers and acquisitions (M&A) deals. M&A deal structuring means deciding whether buying another company will financially benefit the buyer, how to pay for it (using cash or stock), how to keep key employees, and how to handle legal approvals and negotiations. These decisions are important because they determine if the deal will increase profits or create risks for the buyer. My code uses a series of specialized AI agents to automatically analyze financial data, assess risks, and recommend the best deal structure. Instead of manually doing complex financial modeling and scenario planning like a human analyst would, the system takes basic input numbers about both companies and produces recommendations for financing, management retention, legal risks, and negotiation strategies. This type of analysis is normally done by teams of financial analysts, but my AI model streamlines and automates the process.